# SpaceX Falcon 9 - EDA with SQL

Load the wrangled launch dataset into a SQL database and answer standard exploratory questions directly with SQL queries.

In [1]:
import pandas as pd
import sqlite3

con = sqlite3.connect("my_data1.db")
cur = con.cursor()

df = pd.read_csv("dataset_part_1.csv")
df.to_sql("SPACEXTBL", con, if_exists='replace', index=False, method="multi")
pd.read_sql_query("select count(*) as total_rows from SPACEXTBL", con)

,total_rows
0,90


## Unique launch sites in the table

In [2]:
pd.read_sql_query("SELECT DISTINCT LaunchSite FROM SPACEXTBL", con)

,LaunchSite
0,CCAFS SLC 40
1,VAFB SLC 4E
2,KSC LC 39A


## 5 records where launch site begins with 'CCA'

In [3]:
pd.read_sql_query("SELECT * FROM SPACEXTBL WHERE LaunchSite LIKE 'CCA%' LIMIT 5", con)

,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,0,0,0,None,1.0,0,B0003,-80.577366,28.561857
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,0,0,0,None,1.0,0,B0005,-80.577366,28.561857
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,0,0,0,None,1.0,0,B0007,-80.577366,28.561857
3,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,0,0,0,None,1.0,0,B1004,-80.577366,28.561857
4,6,2014-01-06,Falcon 9,3325.000000,GTO,CCAFS SLC 40,None None,1,0,0,0,None,1.0,0,B1005,-80.577366,28.561857


## Total payload mass carried by boosters launched from CCAFS SLC 40

In [4]:
pd.read_sql_query('''
SELECT SUM(PayloadMass) AS Total_Payload_Mass
FROM SPACEXTBL
WHERE LaunchSite = "CCAFS SLC 40"
''', con)

,Total_Payload_Mass
0,305151.428235


## Average payload mass carried by booster version F9 v1.1

In [5]:
pd.read_sql_query('''
SELECT AVG(PayloadMass) AS Avg_Payload_Mass
FROM SPACEXTBL
WHERE BoosterVersion LIKE "%F9 v1.1%"
''', con)

,Avg_Payload_Mass
0,None


## Total number of successful and unsuccessful landing outcomes

In [6]:
pd.read_sql_query('''
SELECT Outcome, COUNT(*) AS Outcome_Count
FROM SPACEXTBL
GROUP BY Outcome
ORDER BY Outcome_Count DESC
''', con)

,Outcome,Outcome_Count
0,True ASDS,41
1,None None,19
2,True RTLS,14
3,False ASDS,6
4,True Ocean,5
5,None ASDS,2
6,False Ocean,2
7,False RTLS,1


## Booster versions which have carried the maximum payload mass

In [7]:
pd.read_sql_query('''
SELECT DISTINCT BoosterVersion
FROM SPACEXTBL
WHERE PayloadMass = (SELECT MAX(PayloadMass) FROM SPACEXTBL)
''', con)

,BoosterVersion
0,Falcon 9


## Launch site totals and success rate per site

In [8]:
pd.read_sql_query('''
SELECT LaunchSite,
       COUNT(*) AS Total_Launches,
       SUM(CASE WHEN Outcome LIKE "True%" THEN 1 ELSE 0 END) AS Successful_Landings,
       ROUND(100.0 * SUM(CASE WHEN Outcome LIKE "True%" THEN 1 ELSE 0 END) / COUNT(*), 1) AS Success_Rate_Pct
FROM SPACEXTBL
GROUP BY LaunchSite
ORDER BY Success_Rate_Pct DESC
''', con)

,LaunchSite,Total_Launches,Successful_Landings,Success_Rate_Pct
0,KSC LC 39A,22,17,77.3
1,VAFB SLC 4E,13,10,76.9
2,CCAFS SLC 40,55,33,60.0


## Orbit types ordered by frequency (excluding GTO transfer orbit)

In [9]:
pd.read_sql_query('''
SELECT Orbit, COUNT(*) AS Launch_Count
FROM SPACEXTBL
WHERE Orbit != "GTO"
GROUP BY Orbit
ORDER BY Launch_Count DESC
''', con)

,Orbit,Launch_Count
0,ISS,21
1,VLEO,14
2,PO,9
3,LEO,7
4,SSO,5
5,MEO,3
6,SO,1
7,HEO,1
8,GEO,1
9,ES-L1,1


## Drone-ship (ASDS) successful landings

In [10]:
pd.read_sql_query('''
SELECT COUNT(*) AS Drone_Ship_Successes
FROM SPACEXTBL
WHERE Outcome = "True ASDS"
''', con)

,Drone_Ship_Successes
0,41
